# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, min_time, max_time, freq_at, size, avg_freq_change, run_likelihood, n_transient

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
# clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n500_r15_y6', 'r36_rp5_mimic', 'n500_r1_y6', 'r36_rp15', 'n1500_r36_y6', 'r36_rp1.5', 'r36_rp12', 'n500_r15_y9', 'r36_rp3.5_mimic', 'n1000_r1_y3', 'r18_rp3.5_mimic', 'r36_rp7_mimic', 'r36_rp13', 'n500_r1_y9', 'r36_rp14', 'n1500_r36_y9', 'r36_rp25', 'n1000_r15_y6', 'r36_rp2.5_mimic', 'r36_rp6', 'n1000_r0_y3', 'r36_rp3_mimic', 'r36_rp1', 'r36_rp8', 'r18_rp2.5_mimic', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y3', 'r36_rp9', 'n1000_r15_y9', 'r36_rp0', 'r36_rp7', 'n500_r0_y6', 'n500_r4_y9', 'r36_rp25_mimic', 'r18_rp8_mimic', 'r36_rp1_mimic', 'n1000_r36_y3', 'r18_rp0.5_mimic', 'n1500_r4_y9', 'r36_rp0.5_mimic', 'n1500_r0_y6', 'r36_rp2.5', 'r36_rp6_mimic', 'r36_rp4_mimic', 'r36_rp20_mimic', 'n1500_r0_y9', 'n1500_r4_y6', 'n500_r36_y3', 'r36_rp36', 'n1500_r1_y9', 'r36_rp0_mimic', 'n1500_r15_y3', 'r18_rp9_mimic', 'r36_rp30', 'r18_rp1.5_mimic', 'r36_rp2_mimic', 'n1500_r1_y6', 'r36_rp1.5_mimic', 'r36_rp3.5', 'r36_rp20', 'r36_rp11', 'r18_rp4.5_mimic', 'r18_rp13_mimic', 'r18_rp7_m

In [4]:
'r36_rp_5_mimic' 

'r36_rp_5_mimic'

In [5]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

for i, df in enumerate(long_dfs):
    if "duplicate" not in df.columns:
        df["duplicate"] = 0
        long_dfs[i] = df

In [6]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'duplicate', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 118800.


init_time  init_freq       r_opt  \
scenario    fish duplicate clone                                     
n500_r15_y6 0    0         0            6.0   0.423404   66.333357   
                           1            6.0   0.331206   28.356251   
                           2            6.0   0.175887   24.619601   
                           3            6.0   0.068794  881.235340   
                           4            6.0   0.000709  263.848310   

                                       l_max      lower       upper  \
scenario    fish duplicate clone                                      
n500_r15_y6 0    0         0      -66.968960  24.140822  215.679403   
                           1     -112.704281  11.747374   84.913303   
                           2     -106.026792  11.252589   59.327711   
                           3      -16.117315  11.652268         NaN   
                           4       -8.517564   0.089875         NaN   

                                       l_null  
scenario    fish duplicate clone               
n500_r15_y6 0    0         0     -1572.903572  
                           1      -761.535855  
                           2      -472.252913  
                           3      -222.795181  
                           4       -18.948396

In [7]:
# Specify order of columns
columns = ['scenario', 'fish', 'duplicate', 'clone', 'init_time', 'init_freq', 'last_time',
           'min_time', 'max_time', 'max_freq', 'n_transient', 'size', 'avg_freq_change', 'r_opt',
           'l_max', 'lower', 'upper', 'l_null']

rows = []
for (scenario, fish, duplicate, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, duplicate, clone) in long_results.index:
            row = long_results.loc[scenario, fish, duplicate, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['duplicate'] = duplicate
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['min_time'] = min_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['n_transient'] = n_transient(data)
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|██████████████████████████████████████| 164547/164547 [04:12<00:00, 650.81it/s]


In [8]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [9]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')

0 rows have r_opt == NA, and 0 have l_max < -1e5 (indicating a failure to converge).


In [10]:
df[df['scenario'] == 'full_pipeline_MP0.95']

,scenario,fish,duplicate,clone,init_time,init_freq,last_time,min_time,max_time,max_freq,n_transient,size,avg_freq_change,r_opt,l_max,lower,upper,l_null
75725,full_pipeline_MP0.95,1,0,0,4.86,0.771956,5.76,4.86,20.53,0.000000,2,0.139727,0.110433,301.628566,-30.123250,75.651422,1445.353071,-6750.366604
75726,full_pipeline_MP0.95,1,0,1,4.86,0.171956,20.53,4.86,20.53,0.802511,11,0.722757,-0.040240,29.705203,-156.524181,14.247565,72.377445,-2399.074473
75727,full_pipeline_MP0.95,1,0,2,4.86,0.020664,5.76,4.86,20.53,0.000000,2,0.004496,-0.005719,12.794846,-20.973270,2.038036,122.295501,-126.778369
75728,full_pipeline_MP0.95,1,0,3,4.86,0.016236,20.53,4.86,20.53,0.068493,11,0.046600,-0.003335,4.974975,-112.798582,2.113326,12.446479,-182.717812
75729,full_pipeline_MP0.95,1,0,4,4.86,0.011808,18.40,4.86,20.53,0.000000,10,0.025762,-0.000365,3.887680,-90.577776,1.510960,10.309603,-133.345874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75993,full_pipeline_MP0.95,8,0,6,4.86,0.073841,12.10,4.86,12.10,0.044060,6,0.051650,0.004113,0.425610,-53.310934,0.067251,2.693354,-58.789496
75994,full_pipeline_MP0.95,8,0,7,4.86,0.039277,12.10,4.86,12.10,0.019670,6,0.030963,0.002708,0.393552,-53.955916,0.033703,3.052544,-57.349902
75995,full_pipeline_MP0.95,8,0,8,4.86,0.025137,12.10,4.86,12.10,0.022030,6,0.020822,0.000429,0.241647,-48.870657,0.000000,3.250449,-49.068883
75996,full_pipeline_MP0.95,8,0,9,4.86,0.014140,12.10,4.86,12.10,0.008655,6,0.008248,0.000758,0.499980,-42.140407,0.000000,2.965496,-43.943218
